# シミュレーション環境の構築

## 前提

- macOS か Windows を使用

## GitとGit LFSのインストール

```sh
# 🍎 macOS
# Homebrewをインストール
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
echo 'eval "$(/opt/homebrew/bin/brew shellenv)"' >> ~/.zprofile
eval "$(/opt/homebrew/bin/brew shellenv)"
brew --version
brew install git git-lfs
git lfs install
```

```sh
# 🪟 Windows
# Git for Windowsをインストール https://git-scm.com/install/windows
git lfs install
```

## UVのインストール

```sh
# 🍎 macOS
curl -LsSf https://astral.sh/uv/install.sh | sh

# 🪟 Windows
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# インストール後ターミナルを開き直し、検証
uv --version
# v0.11.18 古い場合は uv self update を実行
# もしくは curl -LsSf https://astral.sh/uv/install.sh | sh を再実行
```

## 仮想環境を構築

```sh
# 仮想環境を作成
uv venv .reachy_mini_env --python 3.12

# macOS
source .reachy_mini_env/bin/activate

# Windows
# Win + R を押し、powershellと入力。Windows Powershellを右クリックし、管理者として実行
powershell Set-ExecutionPolicy RemoteSigned
# Powershellを閉じ、通常のターミナルを開く
reachy_mini_env\Scripts\activate
```

## Reachy Mini SDK をインストール

https://github.com/pollen-robotics/reachy_mini

In [ ]:
import os

if not os.path.exists("reachy_mini"):
    !git clone https://github.com/pollen-robotics/reachy_mini

!cd reachy_mini && git describe --tags # v1.9.0-91-g234a978e4

In [ ]:
!uv pip install "reachy-mini[mujoco]"

In [ ]:
!uv pip show reachy-mini # 1.11.0.dev0

## シミュレーション環境の実行

仮想環境を有効化

```sh
# macOS
source .reachy_mini_env/bin/activate

# Windows
reachy_mini_env\Scripts\activate
```

シミュレータを起動する。`Ctrl + c`で停止が可能。

```sh
# macOS
mjpython -m reachy_mini.daemon.app.main --sim

# Windows
reachy-mini-daemon --sim
```

シーンを適用したシミュレータを起動する。

```sh
# macOS
mjpython -m reachy_mini.daemon.app.main --sim --scene minimal

# Windows
reachy-mini-daemon --sim --scene minimal
```

シミュレータが起動した状態でロボットを動かす。

In [ ]:
from reachy_mini import ReachyMini 
from reachy_mini.utils import create_head_pose 

# localhost で実行中のシミュレーションに接続
with ReachyMini() as mini: 
    print("シミュレーションに接続しました！") 
    
    # 見上げて首を傾ける
    print("頭部を移動中...") 
    mini.goto_target(
        head=create_head_pose(z=20, roll=10, mm=True, degrees=True),
        duration=1.0 
    ) 
    
    # アンテナを動かす
    print("アンテナを動かしています...") 
    mini.goto_target(antennas=[0.6, -0.6], duration=0.3)
    mini.goto_target(antennas=[-0.6, 0.6], duration=0.3) 
    
    # 初期位置に戻す
    mini.goto_target(
        head=create_head_pose(),
        antennas=[0, 0],
        duration=1.0 
    )
    print("初期位置に戻しました！")

## GStreamerの確認

GStreamerは、マイク・スピーカー・カメラと連携するためのライブラリ。

新しいターミナルを開き、仮想環境を有効化する。

```sh
# macOS
source .reachy_mini_env/bin/activate

# Windows
reachy_mini_env\Scripts\activate
```

GStreamerのバージョンを確認する。

```sh
gst-inspect-1.0 --version # GStreamer 1.28.3
```

## GStreamerの初期化

In [ ]:
import gi
gi.require_version("Gst", "1.0")
from gi.repository import Gst

Gst.init(None)
print("GStreamer:", Gst.version_string(), "\n")

プラグインが正しくインストールされているかを確認する。

In [ ]:
REQUIRED = {
    "webrtcsink":        "WebRTC配信（daemon → 遠隔クライアント）",
    "webrtcbin":         "WebRTC下位実装",
    "webrtcdsp":         "エコーキャンセル(AEC)",
    "webrtcechoprobe":   "AEC参照信号プローブ",
    "unixfdsink":        "IPC送信（LOCALバックエンド, mac/Linux）",
    "unixfdsrc":         "IPC受信（LOCALバックエンド, mac/Linux）",
    "equalizer-10bands": "スピーカーEQ",
    "audiodynamic":      "EQ後のリミッタ",
    "appsink":           "GStreamer → numpy",
    "appsrc":            "numpy → GStreamer",
    "jpegdec":           "USBカメラのMJPEGデコード",
}

for name, desc in REQUIRED.items():
    f = Gst.ElementFactory.find(name)
    mark = "✅" if f else "❌"
    plugin = f" (plugin: {f.get_plugin_name()})" if f else ""
    print(f"{mark} {name:<18} {desc}{plugin}")

## デバイスの確認

利用可能な入力音声デバイス一覧（audio source）を表示する。

In [ ]:
from reachy_mini.media.device_detection import gst_monitor_devices

for d in gst_monitor_devices("Audio/Source"):
    print(f"[{d.index}] {d.display_name}")
    print(f"class={d.device_class}  props={dict(list(d.properties.items())[:4])}")

利用可能な出力音声デバイス一覧（audio sink）を表示する。

In [ ]:
for d in gst_monitor_devices("Audio/Sink"):
    print(f"[{d.index}] {d.display_name}")
    print(f"class={d.device_class}  props={dict(list(d.properties.items())[:4])}")

利用可能なカメラデバイス一覧（video source）を表示する。

In [ ]:
for d in gst_monitor_devices("Video/Source"):
    print(f"[{d.index}] {d.display_name}")
    print(f"class={d.device_class}  props={dict(list(d.properties.items())[:4])}")

ロボットが使用しているデバイスを表示する。

In [ ]:
from reachy_mini.media.device_detection import get_audio_device, get_video_device

mic_id   = get_audio_device("Source")
spk_id   = get_audio_device("Sink")
cam_path, cam_specs = get_video_device()

print(f"Microphone : {mic_id or '(not found / sim モードでは未使用)'}")
print(f"Speaker    : {spk_id or '(not found / sim モードでは未使用)'}")
print(f"Camera     : {cam_path or '(not found — sim モードは MuJoCo UDP 経由)'}")

In [ ]:
# 現在のオーディオ音量を確認

from reachy_mini.daemon.app.routers.volume_control import get_volume_control

vc = get_volume_control()

output_vol = vc.get_output_volume()
input_vol  = vc.get_input_volume()

print(f"プラットフォーム : {vc.platform_name}")
print(f"スピーカー  ({vc.output_device.name}): {output_vol}%")
print(f"マイク      ({vc.input_device.name}) : {input_vol}%")


In [ ]:
# スピーカーとマイクの音量を100%に設定

from reachy_mini.daemon.app.routers.volume_control import get_volume_control

vc = get_volume_control()

ok_out = vc.set_output_volume(100)
ok_in  = vc.set_input_volume(100)

print(f"スピーカー  : {'✓ 100%' if ok_out else '✗ 設定失敗'}")
print(f"マイク      : {'✓ 100%' if ok_in  else '✗ 設定失敗'}")

# 確認
print(f"\n設定後 — スピーカー: {vc.get_output_volume()}%  マイク: {vc.get_input_volume()}%")


In [ ]:
if cam_specs:
    print(f"カメラの仕様: {cam_specs}")
else:
    print("カメラの仕様は取得できませんでした。")

In [ ]:
# カメラの解像度とフレームレートの一覧

from reachy_mini.media.camera_constants import (
    MujocoCameraSpecs,
    ReachyMiniLiteCamSpecs,
    ReachyMiniWirelessCamSpecs,
    ArducamSpecs,
)

for specs_cls in [MujocoCameraSpecs, ReachyMiniLiteCamSpecs, ReachyMiniWirelessCamSpecs, ArducamSpecs]:
    specs = specs_cls()
    print(f"\n【{specs.name}】 デフォルト: {specs.default_resolution.name}")
    for r in specs.available_resolutions:
        w, h, fps, _ = r.value
        marker = " ← default" if r == specs.default_resolution else ""
        print(f"  {w:4d}x{h:<4d} @{fps:2d}fps  ({r.name}){marker}")


## 音声の録音

音声を録音する。

In [ ]:
from reachy_mini import ReachyMini
from scipy.signal import resample
import time
import numpy as np

with ReachyMini(media_backend="default") as mini:

    # 録音パイプラインを開始
    mini.media.start_recording()
    record_seconds = 5

    # マイクのサンプリングレート（Hz）を取得
    in_sr  = mini.media.get_input_audio_samplerate()

    # マイクのチャンネル数を取得
    in_ch  = mini.media.get_input_channels()

    # 録音するサンプル数を計算
    target_samples = int(record_seconds * in_sr)

    # スピーカーのサンプリングレート（Hz）を取得
    out_sr = mini.media.get_output_audio_samplerate()

    print(f"音声を{record_seconds}秒間録音中... ({in_sr} Hz, {in_ch}ch)")

    audio_samples = []
    collected = 0

    while collected < target_samples:
        samples = mini.media.get_audio_sample()
        if samples is not None:
            audio_samples.append(samples)
            collected += len(samples)
            print(f"\rサンプル取得中: {collected/in_sr:.1f}s", end="")
        else:
            time.sleep(0.01)
    print()
    print(f"{len(audio_samples)} 件のサンプルを取得しました。")

    # 録音パイプラインを停止
    mini.media.stop_recording()

audio_data = np.concatenate(audio_samples, axis=0)[:target_samples]
print(f"Recorded audio: {len(audio_data)} samples at {in_sr} Hz, {in_ch}ch")

In [ ]:
# 録音データの検証

x = audio_data.astype(np.float64)

def to_db(v):
    return 20 * np.log10(v) if v > 0 else float("-inf")

peak = float(np.abs(x).max())
rms  = float(np.sqrt(np.mean(x**2)))

print(f"shape       : {audio_data.shape}   dtype: {audio_data.dtype}")
print(f"長さ         : {len(x)/in_sr:.2f} 秒")
print(f"ピーク       : {peak:.6f}  ({to_db(peak):+.1f} dBFS)")
print(f"RMS          : {rms:.6f}  ({to_db(rms):+.1f} dBFS)")
print(f"非ゼロ比率   : {np.count_nonzero(x) / x.size:.1%}")
print(f"NaN / Inf    : {np.isnan(x).any()} / {np.isinf(x).any()}")

for ch in range(x.shape[1]):
    c = x[:, ch]
    print(f"  ch{ch}: peak={np.abs(c).max():.6f}  rms={np.sqrt(np.mean(c**2)):.6f}")

print()
if np.isnan(x).any() or np.isinf(x).any():
    print("❌ NaN / Inf が含まれています")
elif peak == 0.0:
    print("❌ 完全な無音（全サンプルが厳密に 0）→ マイクからデータが届いていません")
elif peak < 1e-3:
    print("⚠️  ほぼ無音（-60 dBFS 未満）→ ゲイン不足か別デバイスの可能性")
elif peak >= 1.0:
    print("⚠️  クリップしています（音が割れます）")
else:
    print("✅ 音声が録れています")


In [ ]:
from IPython.display import Audio

mono = audio_data.mean(axis=1)
display(Audio(data=mono, rate=out_sr))

## 音声の再生

音声をスピーカーで再生する。

In [ ]:
with ReachyMini(media_backend="default") as mini:
    out_sr = mini.media.get_output_audio_samplerate()

    data = audio_data
    # 入出力でレートが違う場合だけリサンプリング（現状はどちらも16kHzなので通らない）
    if in_sr != out_sr:
        data = resample(data, int(len(data) * out_sr / in_sr))

    # push_audio_sample は float32 を要求する（resample は float64 を返すため必須）
    data = np.ascontiguousarray(data, dtype=np.float32)

    # 再生パイプラインを開始
    mini.media.start_playing()
    print(f"再生中... ({len(data)/out_sr:.1f}s, {out_sr} Hz)")

    # チャンクに分けて送出
    chunk_size = 1024
    for i in range(0, len(data), chunk_size):
        mini.media.push_audio_sample(data[i : i + chunk_size])

    # 再生完了を待つ（push は非同期なので待たないと途中で切れる）
    time.sleep(len(data) / out_sr + 0.3)

    # 再生パイプラインを停止
    mini.media.stop_playing()
    print("再生完了")


音声をWAVに保存する。

In [ ]:
import soundfile as sf
from reachy_mini.media.gstreamer_utils import audio_duration_seconds

path = "recorded.wav"
sf.write(path, audio_data, in_sr)   # in_sr で書けばファイル側にレート情報が入る


WAVを再生する。

In [ ]:
import os

with ReachyMini(media_backend="default") as mini:
    mini.media.play_sound(os.path.abspath(path))   # 絶対パスで渡す
    time.sleep(audio_duration_seconds(path) + 0.3)


## カメラで撮影する

In [ ]:
!uv pip install matplotlib

In [ ]:
from reachy_mini import ReachyMini
import matplotlib.pyplot as plt

mini = ReachyMini(media_backend="default")  # with を使わず開いたままにする

# 以降、音声・カメラなど複数セルでこの mini を使い回す
frame = mini.media.get_frame()
plt.imshow(frame[:, :, ::-1])
plt.axis("off")
plt.title(f"frame {frame.shape}, {frame.dtype}")
plt.show()
